In [1]:
import pandas as pd 
import numpy as np

In [2]:
df = pd.read_csv(r"C:\Users\Hp\Downloads\Edukron\Home\Data\application_train.csv")

In [3]:
df.shape

(307511, 122)

In [4]:
df.head()

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


In [5]:
df.isnull().sum()

SK_ID_CURR                        0
TARGET                            0
NAME_CONTRACT_TYPE                0
CODE_GENDER                       0
FLAG_OWN_CAR                      0
                              ...  
AMT_REQ_CREDIT_BUREAU_DAY     41519
AMT_REQ_CREDIT_BUREAU_WEEK    41519
AMT_REQ_CREDIT_BUREAU_MON     41519
AMT_REQ_CREDIT_BUREAU_QRT     41519
AMT_REQ_CREDIT_BUREAU_YEAR    41519
Length: 122, dtype: int64

In [6]:
null_percentage = df.isnull().mean() * 100
null_percentage

SK_ID_CURR                     0.000000
TARGET                         0.000000
NAME_CONTRACT_TYPE             0.000000
CODE_GENDER                    0.000000
FLAG_OWN_CAR                   0.000000
                                ...    
AMT_REQ_CREDIT_BUREAU_DAY     13.501631
AMT_REQ_CREDIT_BUREAU_WEEK    13.501631
AMT_REQ_CREDIT_BUREAU_MON     13.501631
AMT_REQ_CREDIT_BUREAU_QRT     13.501631
AMT_REQ_CREDIT_BUREAU_YEAR    13.501631
Length: 122, dtype: float64

In [7]:
df = df.loc[:, df.isnull().mean() <= 0.30]

In [8]:
df.isnull().sum()

SK_ID_CURR                        0
TARGET                            0
NAME_CONTRACT_TYPE                0
CODE_GENDER                       0
FLAG_OWN_CAR                      0
                              ...  
AMT_REQ_CREDIT_BUREAU_DAY     41519
AMT_REQ_CREDIT_BUREAU_WEEK    41519
AMT_REQ_CREDIT_BUREAU_MON     41519
AMT_REQ_CREDIT_BUREAU_QRT     41519
AMT_REQ_CREDIT_BUREAU_YEAR    41519
Length: 72, dtype: int64

In [9]:
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns

for col in numerical_cols:
    df[col] = df[col].fillna(df[col].median())

In [10]:
categorical_cols = df.select_dtypes(include=['object']).columns

for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

In [11]:
df.isnull().sum().sum()

np.int64(0)

In [43]:
from sklearn.preprocessing import LabelEncoder

categorical_cols = df.select_dtypes(include=['object']).columns

label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

print("Label encoding completed")

Label encoding completed


In [13]:
X = df.drop(['TARGET', 'SK_ID_CURR'], axis=1)
y = df['TARGET']

In [14]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [15]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_train, y_train = smote.fit_resample(X_train, y_train)

In [16]:
print(y_train.value_counts())

TARGET
0    226148
1    226148
Name: count, dtype: int64


In [17]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

DTC = DecisionTreeClassifier(random_state=42)

DTC.fit(X_train, y_train)

y_pred_DTC = DTC.predict(X_test)

dtc_accuracy = accuracy_score(y_test, y_pred_DTC)

print("Decision Tree Accuracy:", dtc_accuracy)

Decision Tree Accuracy: 0.8288863957855714


In [18]:
from sklearn.ensemble import RandomForestClassifier

RFC = RandomForestClassifier(
    random_state=42,
    n_estimators=100
)

RFC.fit(X_train, y_train)

y_pred_RFC = RFC.predict(X_test)

rfc_accuracy = accuracy_score(y_test, y_pred_RFC)

print("Random Forest Accuracy:", rfc_accuracy)

Random Forest Accuracy: 0.9131912264442384


In [19]:
from sklearn.ensemble import BaggingClassifier

BC = BaggingClassifier(
    n_estimators=100,
    random_state=42
)

BC.fit(X_train, y_train)

y_pred_BC = BC.predict(X_test)

bc_accuracy = accuracy_score(y_test, y_pred_BC)

print("Bagging Classifier Accuracy:", bc_accuracy)

Bagging Classifier Accuracy: 0.9162805066419524


In [20]:
results = pd.DataFrame({
    "Model": [
        "Decision Tree",
        "Random Forest",
        "Bagging Classifier"
    ],
    "Accuracy": [
        dtc_accuracy,
        rfc_accuracy,
        bc_accuracy
    ]
})

results

,Model,Accuracy
0,Decision Tree,0.828886
1,Random Forest,0.913191
2,Bagging Classifier,0.916281


In [21]:
from sklearn.feature_selection import SelectKBest, f_classif

selector = SelectKBest(
    score_func=f_classif,
    k=20
)

X_train_fe = selector.fit_transform(X_train, y_train)

X_test_fe = selector.transform(X_test)

In [22]:
DTC_FE = DecisionTreeClassifier(random_state=42)

DTC_FE.fit(X_train_fe, y_train)

y_pred_DTC_FE = DTC_FE.predict(X_test_fe)

dtc_fe_accuracy = accuracy_score(y_test, y_pred_DTC_FE)

print("Feature Engineering DTC Accuracy:", dtc_fe_accuracy)

Feature Engineering DTC Accuracy: 0.7309074354096549


In [23]:
RFC_FE = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

RFC_FE.fit(X_train_fe, y_train)

y_pred_RFC_FE = RFC_FE.predict(X_test_fe)

rfc_fe_accuracy = accuracy_score(y_test, y_pred_RFC_FE)

print("Feature Engineering RFC Accuracy:", rfc_fe_accuracy)

Feature Engineering RFC Accuracy: 0.8142367039006227


In [24]:
BC_FE = BaggingClassifier(
    n_estimators=100,
    random_state=42
)

BC_FE.fit(X_train_fe, y_train)

y_pred_BC_FE = BC_FE.predict(X_test_fe)

bc_fe_accuracy = accuracy_score(y_test, y_pred_BC_FE)

print("Feature Engineering BC Accuracy:", bc_fe_accuracy)

Feature Engineering BC Accuracy: 0.8085134058501211


In [26]:
print("After Feature Engineering Accuracy:")
print("Feature Engineering DTC Accuracy:", dtc_fe_accuracy)
print("Feature Engineering RFC Accuracy:", rfc_fe_accuracy)
print("Feature Engineering BC Accuracy:", bc_fe_accuracy)


After Feature Engineering Accuracy:
Feature Engineering DTC Accuracy: 0.7309074354096549
Feature Engineering RFC Accuracy: 0.8142367039006227
Feature Engineering BC Accuracy: 0.8085134058501211


In [28]:
print("After Feature Engineering Accuracy:")
results = pd.DataFrame({
    "Model": [
        "Decision Tree",
        "Random Forest",
        "Bagging Classifier"
    ],
    "Accuracy": [
        dtc_fe_accuracy,
        rfc_fe_accuracy,
        bc_fe_accuracy
    ]
})

results

After Feature Engineering Accuracy:


,Model,Accuracy
0,Decision Tree,0.730907
1,Random Forest,0.814237
2,Bagging Classifier,0.808513


In [29]:
from sklearn.model_selection import GridSearchCV, KFold

kfold = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [32]:
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

dtc_params = {
    'criterion': ['entropy'],
    'max_depth': [None, 5, 10],
    'min_samples_split': [2, 5]
}

dtc_grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    dtc_params,
    cv=kfold,
    scoring='accuracy',
    n_jobs=-1
)

dtc_grid.fit(X_train, y_train)

dtc_grid_pred = dtc_grid.predict(X_test)

dtc_grid_accuracy = accuracy_score(
    y_test,
    dtc_grid_pred
)

print("Best Parameters:", dtc_grid.best_params_)
print("GridSearch DTC Accuracy:", dtc_grid_accuracy)

Best Parameters: {'criterion': 'entropy', 'max_depth': None, 'min_samples_split': 5}
GridSearch DTC Accuracy: 0.83615433393493


In [33]:
rfc_params = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10],
    'min_samples_split': [2, 5]
}

rfc_grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    rfc_params,
    cv=kfold,
    scoring='accuracy',
    n_jobs=-1
)

rfc_grid.fit(X_train, y_train)

rfc_grid_pred = rfc_grid.predict(X_test)

rfc_grid_accuracy = accuracy_score(
    y_test,
    rfc_grid_pred
)

print("Best Parameters:", rfc_grid.best_params_)
print("GridSearch RFC Accuracy:", rfc_grid_accuracy)

Best Parameters: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 200}
GridSearch RFC Accuracy: 0.9135814513113182


In [34]:
bc_params = {
    'n_estimators': [10, 50, 100],
    'max_samples': [0.5, 1.0],
    'max_features': [0.5, 1.0]
}

bc_grid = GridSearchCV(
    BaggingClassifier(random_state=42),
    bc_params,
    cv=kfold,
    scoring='accuracy',
    n_jobs=-1
)

bc_grid.fit(X_train, y_train)

bc_grid_pred = bc_grid.predict(X_test)

bc_grid_accuracy = accuracy_score(
    y_test,
    bc_grid_pred
)

print("Best Parameters:", bc_grid.best_params_)
print("GridSearch BC Accuracy:", bc_grid_accuracy)

Best Parameters: {'max_features': 0.5, 'max_samples': 1.0, 'n_estimators': 100}
GridSearch BC Accuracy: 0.9192234525145115


In [35]:
results = pd.DataFrame({
    'Model': [
        'Base DTC',
        'Base RFC',
        'Base BC',
        'FE DTC',
        'FE RFC',
        'FE BC',
        'GridSearch DTC',
        'GridSearch RFC',
        'GridSearch BC'
    ],
    
    'Accuracy': [
        dtc_accuracy,
        rfc_accuracy,
        bc_accuracy,
        dtc_fe_accuracy,
        rfc_fe_accuracy,
        bc_fe_accuracy,
        dtc_grid_accuracy,
        rfc_grid_accuracy,
        bc_grid_accuracy
    ]
})

results

,Model,Accuracy
0,Base DTC,0.828886
1,Base RFC,0.913191
2,Base BC,0.916281
3,FE DTC,0.730907
4,FE RFC,0.814237
5,FE BC,0.808513
6,GridSearch DTC,0.836154
7,GridSearch RFC,0.913581
8,GridSearch BC,0.919223


In [36]:
results.sort_values(
    by='Accuracy',
    ascending=False
)

,Model,Accuracy
8,GridSearch BC,0.919223
2,Base BC,0.916281
7,GridSearch RFC,0.913581
1,Base RFC,0.913191
6,GridSearch DTC,0.836154
0,Base DTC,0.828886
4,FE RFC,0.814237
5,FE BC,0.808513
3,FE DTC,0.730907


In [37]:
best_model = results.loc[
    results['Accuracy'].idxmax()
]
print("Best Model:", best_model['Model'])
print("Best Accuracy:", best_model['Accuracy'])

Best Model: GridSearch BC
Best Accuracy: 0.9192234525145115


In [38]:
import pickle

In [39]:
with open("Best model.pkl","wb") as file:
    pickle.dump(best_model,file)
print("Model Saved Successfully")

Model Saved Successfully


In [44]:
import pickle

model_package = {
    'model': bc_grid.best_estimator_,
    'label_encoders': label_encoders,
    'selector': selector
}

pickle_path = r"C:\Users\Hp\Downloads\Edukron\Home\Data\Best_GridSearch_BC.pkl"

with open(pickle_path, "wb") as file:
    pickle.dump(model_package, file)

print("Pickle file saved successfully!")
print(pickle_path)

Pickle file saved successfully!
C:\Users\Hp\Downloads\Edukron\Home\Data\Best_GridSearch_BC.pkl


In [47]:
import pickle

model_package = {
    'model': bc_grid.best_estimator_,
    'selector': selector
}

pickle_path = r"C:\Users\Hp\Downloads\Edukron\Home\Data\Best_GridSearch_BC.pkl"

with open(pickle_path, "wb") as file:
    pickle.dump(model_package, file)

print("Pickle file saved successfully!")

Pickle file saved successfully!


In [48]:
import pandas as pd

test_path = r"C:\Users\Hp\Downloads\Edukron\Home\Data\application_test.xlsx"

test_df = pd.read_excel(test_path)

print(test_df.shape)
test_df.head()

(48744, 123)


,SK_ID_CURR,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,...,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR,AMT_REQ_CREDIT_BUREAU_YEAR.1,Unnamed: 122
0,100001,Cash loans,F,N,Y,0,135000.0,568800.0,20560.5,450000.0,...,0,0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,2.0000
1,100005,Cash loans,M,N,Y,0,99000.0,222768.0,17370.0,180000.0,...,0,0,0.0,0.0,0.0,0.0,0.0,3.0,NaN,2.0000
2,100013,Cash loans,M,Y,Y,0,202500.0,663264.0,69777.0,630000.0,...,0,0,0.0,0.0,0.0,0.0,1.0,4.0,NaN,2.0000
3,100028,Cash loans,F,N,Y,2,315000.0,1575000.0,49018.5,1575000.0,...,0,0,0.0,0.0,0.0,0.0,0.0,3.0,NaN,26.0000
4,100038,Cash loans,M,Y,N,1,180000.0,625500.0,32067.0,625500.0,...,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,516740.4356


In [49]:
from sklearn.preprocessing import LabelEncoder

test_df = test_df.copy()

for col in test_df.select_dtypes(include='object').columns:
    le = LabelEncoder()
    test_df[col] = le.fit_transform(test_df[col].astype(str))

print("Test Label Encoding completed")

Test Label Encoding completed


In [50]:
test_ids = test_df['SK_ID_CURR']

X_test_final = test_df.drop(
    ['SK_ID_CURR'],
    axis=1
)

print(X_test_final.shape)

(48744, 122)


In [51]:
with open(pickle_path, "rb") as file:
    model_package = pickle.load(file)

model = model_package['model']
selector = model_package['selector']

print("Pickle model loaded successfully!")

Pickle model loaded successfully!


In [53]:
print("Features expected by selector:", len(selector.feature_names_in_))
print(selector.feature_names_in_)

Features expected by selector: 70
['NAME_CONTRACT_TYPE' 'CODE_GENDER' 'FLAG_OWN_CAR' 'FLAG_OWN_REALTY'
 'CNT_CHILDREN' 'AMT_INCOME_TOTAL' 'AMT_CREDIT' 'AMT_ANNUITY'
 'AMT_GOODS_PRICE' 'NAME_TYPE_SUITE' 'NAME_INCOME_TYPE'
 'NAME_EDUCATION_TYPE' 'NAME_FAMILY_STATUS' 'NAME_HOUSING_TYPE'
 'REGION_POPULATION_RELATIVE' 'DAYS_BIRTH' 'DAYS_EMPLOYED'
 'DAYS_REGISTRATION' 'DAYS_ID_PUBLISH' 'FLAG_MOBIL' 'FLAG_EMP_PHONE'
 'FLAG_WORK_PHONE' 'FLAG_CONT_MOBILE' 'FLAG_PHONE' 'FLAG_EMAIL'
 'CNT_FAM_MEMBERS' 'REGION_RATING_CLIENT' 'REGION_RATING_CLIENT_W_CITY'
 'WEEKDAY_APPR_PROCESS_START' 'HOUR_APPR_PROCESS_START'
 'REG_REGION_NOT_LIVE_REGION' 'REG_REGION_NOT_WORK_REGION'
 'LIVE_REGION_NOT_WORK_REGION' 'REG_CITY_NOT_LIVE_CITY'
 'REG_CITY_NOT_WORK_CITY' 'LIVE_CITY_NOT_WORK_CITY' 'ORGANIZATION_TYPE'
 'EXT_SOURCE_2' 'EXT_SOURCE_3' 'OBS_30_CNT_SOCIAL_CIRCLE'
 'DEF_30_CNT_SOCIAL_CIRCLE' 'OBS_60_CNT_SOCIAL_CIRCLE'
 'DEF_60_CNT_SOCIAL_CIRCLE' 'DAYS_LAST_PHONE_CHANGE' 'FLAG_DOCUMENT_2'
 'FLAG_DOCUMENT_3' 'FLAG

In [54]:
X_test_final = X_test_final[selector.feature_names_in_]

print("Test shape after matching training features:", X_test_final.shape)

Test shape after matching training features: (48744, 70)


In [56]:
# Fill missing numerical values
num_cols_test = X_test_final.select_dtypes(include=['int64', 'float64']).columns

for col in num_cols_test:
    X_test_final[col] = X_test_final[col].fillna(X_test_final[col].median())

# Check remaining missing values
print("Remaining NaN values:", X_test_final.isnull().sum().sum())

Remaining NaN values: 0


C:\Users\Hp\AppData\Local\Temp\ipykernel_14324\3876307690.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_test_final[col] = X_test_final[col].fillna(X_test_final[col].median())
C:\Users\Hp\AppData\Local\Temp\ipykernel_14324\3876307690.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_test_final[col] = X_test_final[col].fillna(X_test_final[col].median())
C:\Users\Hp\AppData\Local\Temp\ipykernel_14324\3876307690.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a D

In [57]:
X_test_selected = selector.transform(X_test_final)

print("Selected test shape:", X_test_selected.shape)

Selected test shape: (48744, 20)


In [61]:
y_pred = model.predict(X_test_final)

In [62]:
print("Model features:", len(model.feature_names_in_))
print("Test features:", len(X_test_final.columns))

print(model.feature_names_in_)

Model features: 70
Test features: 70
['NAME_CONTRACT_TYPE' 'CODE_GENDER' 'FLAG_OWN_CAR' 'FLAG_OWN_REALTY'
 'CNT_CHILDREN' 'AMT_INCOME_TOTAL' 'AMT_CREDIT' 'AMT_ANNUITY'
 'AMT_GOODS_PRICE' 'NAME_TYPE_SUITE' 'NAME_INCOME_TYPE'
 'NAME_EDUCATION_TYPE' 'NAME_FAMILY_STATUS' 'NAME_HOUSING_TYPE'
 'REGION_POPULATION_RELATIVE' 'DAYS_BIRTH' 'DAYS_EMPLOYED'
 'DAYS_REGISTRATION' 'DAYS_ID_PUBLISH' 'FLAG_MOBIL' 'FLAG_EMP_PHONE'
 'FLAG_WORK_PHONE' 'FLAG_CONT_MOBILE' 'FLAG_PHONE' 'FLAG_EMAIL'
 'CNT_FAM_MEMBERS' 'REGION_RATING_CLIENT' 'REGION_RATING_CLIENT_W_CITY'
 'WEEKDAY_APPR_PROCESS_START' 'HOUR_APPR_PROCESS_START'
 'REG_REGION_NOT_LIVE_REGION' 'REG_REGION_NOT_WORK_REGION'
 'LIVE_REGION_NOT_WORK_REGION' 'REG_CITY_NOT_LIVE_CITY'
 'REG_CITY_NOT_WORK_CITY' 'LIVE_CITY_NOT_WORK_CITY' 'ORGANIZATION_TYPE'
 'EXT_SOURCE_2' 'EXT_SOURCE_3' 'OBS_30_CNT_SOCIAL_CIRCLE'
 'DEF_30_CNT_SOCIAL_CIRCLE' 'OBS_60_CNT_SOCIAL_CIRCLE'
 'DEF_60_CNT_SOCIAL_CIRCLE' 'DAYS_LAST_PHONE_CHANGE' 'FLAG_DOCUMENT_2'
 'FLAG_DOCUMENT_3' 'F

In [63]:
X_test_final = X_test_final[model.feature_names_in_]

In [64]:
y_pred = model.predict(X_test_final)

print(y_pred[:10])
print("Predictions completed")

[0 0 0 0 0 0 0 0 0 0]
Predictions completed


In [65]:
# Do NOT use selector.transform()

X_test_final = X_test_final[model.feature_names_in_]

y_pred = model.predict(X_test_final)

In [67]:
print(y_pred)
print("Total predictions:", len(y_pred))

[0 0 0 ... 0 0 0]
Total predictions: 48744


In [68]:
result = pd.DataFrame({
    'SK_ID_CURR': test_ids,
    'TARGET': y_pred
})

result.to_csv(
    r"C:\Users\Hp\Downloads\Edukron\Home\Data\home_credit_predictions.csv",
    index=False
)

print("Prediction file saved successfully")

Prediction file saved successfully
